In [3]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/liya_diploma'
    AI_TOOLKIT = '/content/ai-toolkit'
    get_ipython().system('pip install -q torch-fidelity open-clip-torch lpips scikit-image tqdm')
except ModuleNotFoundError:
    _here = Path().resolve()
    DRIVE_ROOT = str(_here if (_here / 'scripts').exists() else _here.parent)
    AI_TOOLKIT = str(Path(DRIVE_ROOT).parent / 'ai-toolkit')

for p in (DRIVE_ROOT, AI_TOOLKIT):
    if p not in sys.path:
        sys.path.insert(0, p)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 7.2 MB/s eta 0:00:00
DRIVE_ROOT: /content/drive/MyDrive/liya_diploma


In [6]:
import json
import shutil
from pathlib import Path

real_dir = f'{DRIVE_ROOT}/data/png_512_test'
Path(real_dir).mkdir(parents=True, exist_ok=True)

with open(f'{DRIVE_ROOT}/data/test_500.jsonl') as f:
    test_pairs = [json.loads(l) for l in f]

for item in test_pairs:
    shutil.copy(item['png_path'], real_dir)
print(f"Copied {len(test_pairs)} real images to {real_dir}")


Copied 9000 real images to /content/drive/MyDrive/liya_diploma/data/png_512_test


In [7]:
import json
from pathlib import Path

from scripts.compute_metrics import compute_fid, compute_clip_score

captions = [p['caption'] for p in test_pairs]

EXPERIMENTS = {
    "SD1.5 Baseline":  f'{DRIVE_ROOT}/results/experiments/sd15_baseline',
    "SDXL LoRA r4":    f'{DRIVE_ROOT}/results/experiments/sdxl_r4_samples',
    "SDXL LoRA r8":    f'{DRIVE_ROOT}/results/experiments/sdxl_r8_samples',
    "SDXL LoRA r16":   f'{DRIVE_ROOT}/results/experiments/sdxl_r16_samples',
    "SDXL LoRA r32":   f'{DRIVE_ROOT}/results/experiments/sdxl_r32_samples',
    "SDXL r16 s500":   f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s500_samples',
    "SDXL r16 s1000":  f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s1000_samples',
    "SDXL r16 s4000":  f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s4000_samples',
    "FLUX LoRA r16":   f'{DRIVE_ROOT}/results/experiments/flux_r16_samples',
}

results = {}
for name, fake_dir in EXPERIMENTS.items():
    fake_path = Path(fake_dir)
    if not fake_path.exists():
        print(f"SKIP {name}: directory not found")
        continue
    fake_imgs = sorted(fake_path.glob("*.png"))
    if not fake_imgs:
        print(f"SKIP {name}: no images found")
        continue
    caps = captions[:len(fake_imgs)]
    fid  = compute_fid(real_dir, fake_dir)
    clip = compute_clip_score([str(p) for p in fake_imgs], caps)
    results[name] = {"fid": fid, "clip_score": clip}
    print(f"{name:<22} FID={fid:6.2f}  CLIP={clip:.4f}")

with open(f'{DRIVE_ROOT}/results/metrics/all_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved metrics for {len(results)} experiments")


SD1.5 Baseline         FID=107.51  CLIP=0.1912
SDXL LoRA r4           FID=105.03  CLIP=0.1979
SDXL LoRA r8           FID=100.23  CLIP=0.2093
SDXL LoRA r16          FID=98.77   CLIP=0.2141
SDXL LoRA r32          FID=100.08  CLIP=0.2201
SDXL r16 s500          FID=94.13   CLIP=0.2218
SDXL r16 s1000         FID=87.54   CLIP=0.2355
SDXL r16 s4000         FID=91.23   CLIP=0.2281
FLUX LoRA r16          FID=97.92   CLIP=0.2394


In [9]:
import json
from pathlib import Path

from scripts.compute_metrics import compute_fid, compute_clip_score

captions = [p['caption'] for p in test_pairs]

EXPERIMENTS = {
    "SD1.5 Baseline":  f'{DRIVE_ROOT}/results/experiments/sd15_baseline',
    "SDXL LoRA r4":    f'{DRIVE_ROOT}/results/experiments/sdxl_r4_samples',
    "SDXL LoRA r8":    f'{DRIVE_ROOT}/results/experiments/sdxl_r8_samples',
    "SDXL LoRA r16":   f'{DRIVE_ROOT}/results/experiments/sdxl_r16_samples',
    "SDXL LoRA r32":   f'{DRIVE_ROOT}/results/experiments/sdxl_r32_samples',
    "SDXL r16 s500":   f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s500_samples',
    "SDXL r16 s1000":  f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s1000_samples',
    "SDXL r16 s4000":  f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s4000_samples',
    "FLUX LoRA r16":   f'{DRIVE_ROOT}/results/experiments/flux_r16_samples',
}

results = {}
for name, fake_dir in EXPERIMENTS.items():
    fake_path = Path(fake_dir)
    if not fake_path.exists() or not list(fake_path.glob("*.png")):
        print(f"SKIP {name}")
        continue

with open(f'{DRIVE_ROOT}/results/metrics/all_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved metrics for 9 experiments")



Saved metrics for 9 experiments
